In [2]:
# Cryptarithmetic Puzzle using CSP

letters = ['S', 'E', 'N', 'D', 'M', 'O', 'R', 'Y']

domains = {letter: set(range(10)) for letter in letters}

# Leading zero is not allowed
domains['S'].remove(0)
domains['M'].remove(0)

In [3]:
def check_constraints(assignment):
    # All letters must have unique digits
    if len(set(assignment.values())) != len(assignment):
        return False

    # Leading zero constraint
    if assignment.get('S', 1) == 0:
        return False
    if assignment.get('M', 1) == 0:
        return False

    # Check arithmetic only when all letters are assigned
    if len(assignment) == len(letters):
        SEND = (assignment['S'] * 1000 +
                assignment['E'] * 100 +
                assignment['N'] * 10 +
                assignment['D'])

        MORE = (assignment['M'] * 1000 +
                assignment['O'] * 100 +
                assignment['R'] * 10 +
                assignment['E'])

        MONEY = (assignment['M'] * 10000 +
                 assignment['O'] * 1000 +
                 assignment['N'] * 100 +
                 assignment['E'] * 10 +
                 assignment['Y'])

        return SEND + MORE == MONEY

    return True

In [4]:
# Minimum Remaining Values (MRV)
def select_variable(assignment, current_domains):
    unassigned = [v for v in letters if v not in assignment]

    return min(unassigned, key=lambda v: len(current_domains[v]))


# Least Constraining Value (LCV)
def order_values(variable, assignment, current_domains):
    values = list(current_domains[variable])

    def conflict_count(value):
        conflicts = 0

        for other in letters:
            if other != variable and other not in assignment:
                if value in current_domains[other]:
                    conflicts += 1

        return conflicts

    return sorted(values, key=conflict_count)

In [5]:
# Forward Checking
def forward_check(variable, value, assignment, current_domains):
    new_domains = {v: set(current_domains[v]) for v in letters}

    new_domains[variable] = {value}

    for other in letters:
        if other not in assignment and other != variable:
            new_domains[other].discard(value)

    # Check if any unassigned variable has an empty domain
    for other in letters:
        if other not in assignment and other != variable:
            if not new_domains[other]:
                return None

    return new_domains

In [6]:
# Constraint Propagation
def propagate(assignment, current_domains):
    # Enforce all-different constraint through domain reduction
    changed = True

    while changed:
        changed = False
        assigned_values = set(assignment.values())

        for variable in letters:
            if variable not in assignment:
                reduced = current_domains[variable] - assigned_values

                if reduced != current_domains[variable]:
                    current_domains[variable] = reduced
                    changed = True

                if not current_domains[variable]:
                    return False

    return True

In [7]:
# Backtracking Search
def backtracking(assignment, current_domains):
    if len(assignment) == len(letters):
        if check_constraints(assignment):
            return assignment
        return None

    variable = select_variable(assignment, current_domains)

    for value in order_values(variable, assignment, current_domains):

        if value in assignment.values():
            continue

        assignment[variable] = value

        if check_constraints(assignment):

            new_domains = forward_check(
                variable, value, assignment, current_domains
            )

            if new_domains is not None:

                if propagate(assignment, new_domains):
                    result = backtracking(assignment, new_domains)

                    if result is not None:
                        return result

        # Backtrack
        del assignment[variable]

    return None

In [8]:
# Execute the CSP solver
solution = backtracking({}, domains)

# Display the result
if solution:
    print("Solution found:")

    for letter in letters:
        print(letter, "=", solution[letter])

    SEND = (solution['S'] * 1000 +
            solution['E'] * 100 +
            solution['N'] * 10 +
            solution['D'])

    MORE = (solution['M'] * 1000 +
            solution['O'] * 100 +
            solution['R'] * 10 +
            solution['E'])

    MONEY = (solution['M'] * 10000 +
             solution['O'] * 1000 +
             solution['N'] * 100 +
             solution['E'] * 10 +
             solution['Y'])

    print("\nSEND =", SEND)
    print("MORE =", MORE)
    print("MONEY =", MONEY)

    print("\nVerification:")
    print(SEND, "+", MORE, "=", MONEY)
else:
    print("No solution found.")


Solution found:
S = 9
E = 5
N = 6
D = 7
M = 1
O = 0
R = 8
Y = 2

SEND = 9567
MORE = 1085
MONEY = 10652

Verification:
9567 + 1085 = 10652
